# **🧼🫧🧹 Data Cleaning**

<img src="../assets/data_collection.png" style="width:75%">

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# 📊 Create dummy dataset with common issues
data = {
    'ID': range(1, 11),
    'Name': ['Alice', 'Bob', 'Charlie', 'Dave', 'Eve', 'NULL', 'Grace', 'Heidi', 'Ivan', 'Judy'],
    'Age': ['25', 'thirty', '35', '-40', '22', '27', '26', None, '29', '999'],
    'Gender': ['F', 'M', 'M', 'M', 'F', 'Other', 'F', 'N/A', 'F', 'F'],
    'Income': ['50,000', '60000', '70,000', 'N/A', '40000', '30000', '50000', '45000', None, '2000000'],
    'Signup Date': ['2020/01/01', '2020-02-15', '03-10-2020', None, '2020/05/05', '2020-06-01', '2020-07-07', '2020/08/08', 'invalid', '2020/09/09'],
    'Country': ['US', 'US', 'US', 'UK', 'UK', 'UK', 'DE', 'FR', 'FR', 'SG'],
    'Feedback': ['Good', 'Bad', 'Excellent', 'Poor', 'Good', 'NULL', 'Okay', 'Bad', 'Good', 'Poor']
}

df = pd.DataFrame(data)
print("🔍 Raw Data:")
df.head()

🔍 Raw Data:


,ID,Name,Age,Gender,Income,Signup Date,Country,Feedback
0,1,Alice,25,F,"50,000",2020/01/01,US,Good
1,2,Bob,thirty,M,60000,2020-02-15,US,Bad
2,3,Charlie,35,M,"70,000",03-10-2020,US,Excellent
3,4,Dave,-40,M,N/A,None,UK,Poor
4,5,Eve,22,F,40000,2020/05/05,UK,Good


---
---
# **🧼 General Cleaning**

---
### └─ **Apply snake case to column names**

In [4]:
import re
def convert_column_names_to_snake_case(df):
    print("   └── Converting column names to snake_case...")
    def to_snake_case(text):
        text = text.lower().strip()
        text = re.sub(r'\s+', '_', text)            
        text = re.sub(r'[^a-z0-9_]', '', text)      
        return text
    
    df = df.copy()
    df.columns = [to_snake_case(col) for col in df.columns]
    return df

In [5]:
df_sample = pd.DataFrame({
    'First Name': ['Alice', 'Bob', 'Charlie'],
    'Last Name': ['Smith', 'Jones', 'Brown'],
    'Favorite Color': ['Blue', 'Green', 'Red'],
    'Year of Birth': [1990, 1985, 1978]
})

df_sample = convert_column_names_to_snake_case(df_sample)
df_sample.head()

   └── Converting column names to snake_case...


,first_name,last_name,favorite_color,year_of_birth
0,Alice,Smith,Blue,1990
1,Bob,Jones,Green,1985
2,Charlie,Brown,Red,1978


---
### └─ **Drop duplicates**

In [6]:
def drop_duplicates(df, subset=None):
    if subset:
        print(f"   └── Dropping duplicates based on subset columns: {subset}")
    else:
        print("   └── Dropping duplicates based on all columns...")
    df = df.copy()
    df = df.drop_duplicates(subset=subset)
    return df

In [7]:
df_sample = pd.DataFrame({
    'id': [1, 2, 2, 3, 4, 4, 5],
    'name': ['Alice', 'Bob', 'Bob', 'Charlie', 'David', 'David', 'Eve'],
    'age': [25, 30, 30, 35, 40, 40, 45]
})

df_sample = drop_duplicates(df_sample, subset=['id', 'name'])
df_sample

   └── Dropping duplicates based on subset columns: ['id', 'name']


,id,name,age
0,1,Alice,25
1,2,Bob,30
3,3,Charlie,35
4,4,David,40
6,5,Eve,45


---
### └─ **Mark missing values**

In [8]:
def mark_missing_numerical(df, placeholder_value=999, columns=[]):
    print(f"   └── Imputing missing values in numerical columns: {columns} with placeholder: {placeholder_value}")
    df = df.copy()
    for column in columns:
        if column in df.columns and df[column].isnull().any():
            df[column] = df[column].fillna(placeholder_value)
    return df

In [9]:
def mark_missing_categorical(df, placeholder_value='missing', columns=[]):
    print(f"   └── Imputing missing values in categorical columns: {columns} with placeholder: '{placeholder_value}'")
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype('object')  # ensure consistent dtype
            df[col] = df[col].fillna(placeholder_value)
    return df

In [10]:
df_sample = pd.DataFrame({
    'age': [25, 30, np.nan, 40, np.nan],
    'income': [50000, 60000, 55000, np.nan, 100000],
    'score': [1, np.nan, 3, 4, 5],
    'gender': ['Male', 'Female', None, 'Male', 'Female'],
    'city': ['New York', np.nan, 'Los Angeles', 'Chicago', None]
})

df_sample = mark_missing_numerical(df_sample, placeholder_value=999, columns=['age'])
df_sample = mark_missing_categorical(df_sample, placeholder_value="missing", columns=['gender'])
df_sample

   └── Imputing missing values in numerical columns: ['age'] with placeholder: 999
   └── Imputing missing values in categorical columns: ['gender'] with placeholder: 'missing'


,age,income,score,gender,city
0,25.0,50000.0,1.0,Male,New York
1,30.0,60000.0,NaN,Female,NaN
2,999.0,55000.0,3.0,missing,Los Angeles
3,40.0,NaN,4.0,Male,Chicago
4,999.0,100000.0,5.0,Female,None


---
### └─ **Convert data type**

In [20]:
def convert_columns_dtype(df, columns_types):
    df = df.copy()
    for col, dtype in columns_types.items():
        if col in df.columns:
            print(f"   └── Converting column '{col}' to '{dtype}'")
            if dtype == 'numeric':
                df[col] = pd.to_numeric(df[col], errors='coerce')
            elif dtype == 'category':
                df[col] = df[col].astype('category')
            elif dtype == 'string':
                df[col] = df[col].astype('string')
            elif dtype == 'datetime':
                df[col] = pd.to_datetime(df[col], errors='coerce')
            else:
                # For other types like 'int', 'float', etc.
                try:
                    df[col] = df[col].astype(dtype)
                except Exception as e:
                    print(f"      ! Warning: Could not convert column '{col}' to '{dtype}': {e}")
    return df

In [21]:
df_sample = pd.DataFrame({
    'age': ['25', '30', '35', '40', 'not available'],
    'gender': ['Male', 'Female', 'Female', 'Male', 'Female'],
    'signup_date': ['2023-01-10', '2023-02-15', '2023-03-20', 'not a date', '2023-05-01'],
    'score': ['88.5', '92.3', '85.0', '90.2', '87.7'],
    'user_id': [101, 102, 103, 104, 105]
})

df_sample = convert_columns_dtype(df_sample, {
    'age': 'numeric',
    'gender': 'category',
    'signup_date': 'datetime',
    # 'score': 'float',
    # 'user_id': 'string'
})
df_sample.dtypes

   └── Converting column 'age' to 'numeric'
   └── Converting column 'gender' to 'category'
   └── Converting column 'signup_date' to 'datetime'


age                   float64
gender               category
signup_date    datetime64[ns]
score                  object
user_id                 int64
dtype: object

---
---
# **🧼 Numerical Cleaning**

---
### └─ **Get absolute values**

In [18]:
def get_absolute_values(df, columns=[]):
    print(f"   └── Taking absolute values for columns: {columns}")
    df = df.copy()
    for col in columns:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].abs()
    return df

In [19]:
df_sample = pd.DataFrame({
    'age': [25, -30, 45, -50, 60],
    'income': [-50000, 60000, -55000, 70000, -100000],
    'score': [1, -2, 3, -4, 5],
    'gender': ['Male', 'Female', 'Female', 'Male', 'Male']  # non-numeric column remains unchanged
})

df_sample = get_absolute_values(df_sample, columns=['age', 'score'])
df_sample

   └── Taking absolute values for columns: ['age', 'score']


,age,income,score,gender
0,25,-50000,1,Male
1,30,60000,2,Female
2,45,-55000,3,Female
3,50,70000,4,Male
4,60,-100000,5,Male


---
---
# **🧼 Categorical Cleaning**

---
### └─ **Map values**

In [11]:
def map_values(df, mapping_dict):
    print(f"   └── Mapping values for columns: {list(mapping_dict.keys())}")
    df = df.copy()
    columns = mapping_dict.keys()
    for col in columns:
        if col in df.columns and col in mapping_dict:
            df[col] = df[col].map(mapping_dict[col]).fillna(df[col])
    return df

In [13]:
df_sample = pd.DataFrame({
    'Subscription Status': ['yes', 'no', 'no', 'yes'],
    'Gender': ['Male', 'Female', 'Female', 'Male'],
    'Age': [25, 30, 22, 40]
})

mapping = {
    'Subscription Status': {'yes': 1, 'no': 0},
    'Gender': {'Male': 0, 'Female': 1}
}

df_sample = map_values(df_sample, mapping)
df_sample

   └── Mapping values for columns: ['Subscription Status', 'Gender']


,Subscription Status,Gender,Age
0,1,0,25
1,0,1,30
2,0,1,22
3,1,0,40


---
### └─ **Convert test to lowercase**

In [14]:
def convert_text_to_lowercase(df, columns=[]):
    print(f"   └── Converting text to lowercase for columns: {columns}")
    df = df.copy()
    for col in columns:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = df[col].str.lower()
    return df

In [15]:
df_sample = pd.DataFrame({
    'Name': ['Alice', 'BOB', 'Charlie'],
    'City': ['New York', 'LONDON', 'ToKyo'],
    'Age': [25, 30, 22]
})

df_sample = convert_text_to_lowercase(df_sample, columns=['Name'])
df_sample

   └── Converting text to lowercase for columns: ['Name']


,Name,City,Age
0,alice,New York,25
1,bob,LONDON,30
2,charlie,ToKyo,22


---
### └─ **Replace character**

In [16]:
def replace_character_in_columns(df, to_replace, replacement, columns=[]):
    print(f"   └── Replacing '{to_replace}' with '{replacement}' in columns: {columns}")
    df = df.copy()
    for col in columns:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = df[col].str.replace(to_replace, replacement, regex=False)
    return df

In [17]:
df_sample = pd.DataFrame({
    'name': ['Alice Smith', 'Bob.Jones', 'Charlie.Lane'],
    'city': ['New York', 'Los_Angeles', 'San_Francisco']
})

df_sample = replace_character_in_columns(df_sample, to_replace='.', replacement=' ', columns=['name'])
df_sample

   └── Replacing '.' with ' ' in columns: ['name']


,name,city
0,Alice Smith,New York
1,Bob Jones,Los_Angeles
2,Charlie Lane,San_Francisco


---
---
---